In [ ]:

import torch

from dataset_loaders import build_data_loaders
from utils.checkpoints import load_ae_from_path, load_cspn_from_path, load_from_wandb
from utils.config import DatasetConfig
from utils.visualisation import plot_latent_space, show


In [ ]:
ae_path = load_from_wandb("autoencoder_mnist")
ae = load_ae_from_path(ae_path, device=torch.device("mps"))


In [ ]:
dataset_cfg = DatasetConfig(
    name="mnist",
    channels=3,
    height=28,
    width=28,
    num_classes=10,
)

_, dataloader = build_data_loaders(dataset_cfg, batch_size=64)


In [ ]:
cspn_psi_path = load_from_wandb("cspn_mnist_psinet")
cspn_psi = load_cspn_from_path(cspn_psi_path, device=torch.device("mps"))
cspn_psi.eval()

In [ ]:
label = 0
sample_labels = torch.tensor([label] * 3)

with torch.no_grad():
    samples_psi = cspn_psi.sample(sample_labels)
    sampled_images_psi_logits = ae.decode(samples_psi)
    sampled_images_psi = torch.sigmoid(sampled_images_psi_logits)

show(sampled_images_psi, f"Samples from PSINet CSPN with label {label}")

In [ ]:
all_labels = torch.arange(10, dtype=torch.long)

with torch.no_grad():
    samples_psi = cspn_psi.sample(all_labels)
    sampled_images_psi = ae.decode(samples_psi)

show(sampled_images_psi, "Samples from PSINet CSPN for all labels", width=5)

In [ ]:
labels = torch.arange(10).repeat_interleave(100)
with torch.no_grad():
    samples_psi = cspn_psi.sample(labels)

plot_latent_space(samples_psi, labels, title="Latent Space of PSINet CSPN on MNIST")

In [ ]:
random_latents = torch.randn(10, 16)
with torch.no_grad():
    random_samples_psi_logits = ae.decode(random_latents)
    random_samples_psi = torch.sigmoid(random_samples_psi_logits)

show(random_samples_psi, "Random samples from AE latent space", width=10)

In [ ]:
from torchinfo import summary

summary(ae)
summary(cspn_psi)
print([type(l).__name__ for l in cspn_psi.einet.einet_layers])

In [ ]:
import torch


@torch.no_grad()
def check_class_separation(
        model: "PsiNetCSPN",
        autoencoder: "AbstractAutoencoder",
        images: torch.Tensor,
        true_labels: torch.Tensor,
        num_classes: int,
) -> torch.Tensor:
    """For each image, compute log p(z | y) for every possible y.
    Returns a (batch, num_classes) matrix of log-likelihoods."""
    model.eval()
    latent: torch.Tensor = autoencoder.encode(images)
    batch: int = images.shape[0]
    lls = torch.zeros(batch, num_classes, device=images.device)
    for y in range(num_classes):
        labels: torch.Tensor = torch.full((batch,), y, dtype=torch.long, device=images.device)
        lls[:, y] = model(latent, labels)
    return lls


# usage on a held-out batch:
images, labels = next(iter(dataloader))
lls = check_class_separation(cspn_psi, ae, images, labels, num_classes=10)
pred = lls.argmax(dim=1)
acc = (pred == labels).float().mean()
print("implied classification accuracy from likelihood:", acc.item())
print("mean margin (true-label ll minus best wrong-label ll):",
      (lls.gather(1, labels[:, None]).squeeze(1) - lls.masked_fill(
          torch.nn.functional.one_hot(labels, 10).bool(), float("-inf")).max(1).values).mean().item())

In [ ]:
import torch


@torch.no_grad()
def check_sum_weight_specialization(model, num_classes: int) -> None:
    """Compare reparam'd sum-layer weights across different labels.
    If they're nearly identical, the sum layers aren't specializing by class."""
    einet = model.einet
    labels = torch.arange(num_classes)
    params_per_label = einet.param_nn(labels, None)  # list of tensors, one per layer

    for layer_idx, layer_params in enumerate(params_per_label):
        if layer_idx == 0:
            continue  # leaf params (mu/var), skip -- we expect these to vary
        layer = einet.einet_layers[layer_idx]
        reparam = layer.reparam_function()
        weights = reparam(layer_params)  # (num_classes, *param_shape)
        flat = weights.reshape(num_classes, -1)
        mean_pairwise_l1 = torch.cdist(flat, flat, p=1).mean().item()
        print(f"layer {layer_idx} ({type(layer).__name__}): "
              f"mean pairwise L1 distance across classes = {mean_pairwise_l1:.6f}, "
              f"mean weight magnitude = {flat.abs().mean().item():.6f}")


check_sum_weight_specialization(cspn_psi, num_classes=10)